# Simple Gemini API Experiment

**Goal:** send a prompt to a Gemini language model and read its response using Python.

This notebook is intentionally small. We'll do only four things:
1. Install Google's Gemini Python library.
2. Enter a Gemini API key privately.
3. Send one prompt to the model.
4. Experiment with a few prompts and a simple multi-turn chat.

**Before you start:** create a Gemini API key in [Google AI Studio](https://aistudio.google.com/app/apikey). Keep it private. API access and usage limits depend on your Google account and current API terms.

## 1. Install the Gemini library

`google-genai` is Google's Python library for calling the Gemini API. The `%pip` command installs it in the current Jupyter/Colab environment.

In [ ]:
%pip -q install -U google-genai

## 2. Enter your API key

The API key lets your notebook authenticate with Google's API. `getpass` hides what you type so the key is not displayed in the cell output.

**Do not put the actual key directly in the code** or share a notebook with your key saved in it.

In [ ]:
from getpass import getpass
from google import genai

api_key = getpass("Paste your Gemini API key (input hidden): ").strip()

if not api_key:
    raise ValueError("No API key entered. Run this cell again and enter your key.")

client = genai.Client(api_key=api_key)
print("✅ Gemini client created. Your key was not printed.")

## 3. Send your first prompt

A **prompt** is the instruction or question we send to the model. The client sends it to Gemini through the API. The model generates a response, and `print()` displays that response here.

In [ ]:
model_name = "gemini-2.5-flash"

prompt = "Explain what a large language model is in simple language, using a real-life example."

response = client.models.generate_content(
    model=model_name,
    contents=prompt
)

print(response.text)

### What's happening in that code?

- `model_name` chooses which Gemini model handles the request.
- `prompt` contains the text we want Gemini to respond to.
- `client.models.generate_content(...)` sends the request to Google's Gemini API.
- `response` contains the API result.
- `response.text` is the generated text.

The model does not automatically know your intention beyond the instructions and context you send. Try changing the prompt and run the next cell again.

## 4. Experiment with your own prompt

Change `my_prompt` to anything you want to test, then run the cell. Compare how different instructions change the answer.

In [ ]:
my_prompt = "Teach me recursion in C++ using a simple example and explain it like I am a beginner."

response = client.models.generate_content(
    model=model_name,
    contents=my_prompt
)

print(response.text)

## 5. Give the model a role (system instruction)

A system instruction sets general guidance for the model's response style. It is different from the user's specific question.

In [ ]:
from google.genai import types

response = client.models.generate_content(
    model=model_name,
    contents="Explain APIs.",
    config=types.GenerateContentConfig(
        system_instruction="You are a friendly computer science tutor. Use simple words, one analogy, and a short example."
    )
)

print(response.text)

## 6. A tiny multi-turn chat

For a conversation, the model needs the earlier messages included as context. This simple example keeps a short history in Python and sends it with each new message.

Run the cell, enter a message, and repeat. Type `exit` to stop. This is a basic demonstration, not a full chatbot interface.

In [ ]:
from google.genai import types

chat_history = []

print("Gemini chat started. Type 'exit' to stop.\\n")

while True:
    user_message = input("You: ").strip()

    if user_message.lower() in {"exit", "quit"}:
        print("Chat ended.")
        break

    if not user_message:
        continue

    chat_history.append(types.Content(
        role="user",
        parts=[types.Part(text=user_message)]
    ))

    response = client.models.generate_content(
        model=model_name,
        contents=chat_history
    )

    answer = response.text or "(The model returned no text.)"
    print("\\nGemini:", answer, "\\n")

    chat_history.append(types.Content(
        role="model",
        parts=[types.Part(text=answer)]
    ))

## 7. What to experiment with

Try one change at a time:
- Ask for a shorter or longer answer.
- Request a table, a numbered list, or an analogy.
- Give a role in the system instruction (tutor, interviewer, code reviewer).
- Ask a follow-up in the chat and see how the previous messages affect the answer.

## Troubleshooting

- **401 / unauthenticated:** check that the API key is correct.
- **403 / permission error:** check API access and account/project settings.
- **429 / quota or rate limit:** you may have reached a usage limit; check your current Google AI Studio/API limits.
- **Model not found:** the model name may not be available to your account; check Google's current model list and update `model_name`.
- **No output:** make sure you ran the cell and look below it for the result.

**Security reminder:** don't share your key, commit it to GitHub, or publish a notebook containing it. If you accidentally expose it, revoke it and create a new one.